# Notebook 05 — CP02: Problem Setup and Baselines

**Module:** 20 — Capstone Projects  
**Tier:** 1 — Master  
**Notebook:** 5 of 12  
**Time estimate:** 75 minutes

> CP02 is a full TF binding site prediction study: dataset generation,
> baseline models with proper cross-validation, then a CNN in NB06.
> This is the same problem as Module 14 NB12 but now done rigorously.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings('ignore', category=ConvergenceWarning)

rng = np.random.default_rng(42)

# ---- Dataset: GATAAG motif binding ----
N = 3000; L = 101; MOTIF = 'GATAAG'
NUCL = 'ACGT'

def encode_onehot(seq):
    mapping = {'A':0,'C':1,'G':2,'T':3}
    arr = np.zeros((4, len(seq)), dtype=np.float32)
    for i, c in enumerate(seq): arr[mapping.get(c, 0), i] = 1.0
    return arr

def random_seq(L, rng):
    return ''.join(rng.choice(list(NUCL), L))

def inject_motif(seq, motif, rng):
    pos = rng.integers(0, len(seq) - len(motif))
    return seq[:pos] + motif + seq[pos + len(motif):]

# Positive: motif injected + flanking variation
positives = [inject_motif(random_seq(L, rng), MOTIF, rng) for _ in range(N//2)]
# Negative: GC-matched but no motif
negatives = [random_seq(L, rng) for _ in range(N//2)]

all_seqs   = positives + negatives
labels     = np.array([1]*(N//2) + [0]*(N//2))

# ---- k-mer features (k=4) ----
KMER_K = 4
def kmer_freq_vector(seq, k):
    kmers = [''.join(p) for p in np.ndindex(*[4]*k)]
    kmer_to_idx = {kmer: i for i, kmer in enumerate(set(''.join(c) for c in __import__('itertools').product('ACGT', repeat=k)))}
    vec = np.zeros(4**k, dtype=np.float32)
    for i in range(len(seq)-k+1):
        kmer = seq[i:i+k]
        if kmer in kmer_to_idx: vec[kmer_to_idx[kmer]] += 1
    total = vec.sum()
    if total > 0: vec /= total
    return vec

print('Computing k-mer features (this may take 20-30s)...')
X_kmer = np.vstack([kmer_freq_vector(s, KMER_K) for s in all_seqs])
print(f'k-mer feature matrix: {X_kmer.shape}')

# ---- 5-fold cross-validation ----
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_evaluate(X, y, clf, skf):
    aurocs, auprcs = [], []
    for train_idx, test_idx in skf.split(X, y):
        clf.fit(X[train_idx], y[train_idx])
        proba = clf.predict_proba(X[test_idx])[:, 1]
        aurocs.append(roc_auc_score(y[test_idx], proba))
        auprcs.append(average_precision_score(y[test_idx], proba))
    return np.array(aurocs), np.array(auprcs)

# Baseline 1: k-mer + Logistic Regression
lr_pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(C=1.0, max_iter=200))])
lr_auroc, lr_auprc = cv_evaluate(X_kmer, labels, lr_pipe, skf)

# Baseline 2: k-mer + SVM (RBF kernel)
svm_pipe = Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', C=1.0, probability=True))])
svm_auroc, svm_auprc = cv_evaluate(X_kmer, labels, svm_pipe, skf)

print('\n=== Baseline Results (5-fold CV) ===')
for name, aurocs, auprcs in [('k-mer + LR', lr_auroc, lr_auprc), ('k-mer + SVM', svm_auroc, svm_auprc)]:
    print(f'{name:15}  AUROC={aurocs.mean():.3f}±{aurocs.std():.3f}  AUPRC={auprcs.mean():.3f}±{auprcs.std():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# Panel A: AUROC comparison (violin)
ax = axes[0]
data_auroc = [lr_auroc, svm_auroc]
positions = [1, 2]
parts = ax.violinplot(data_auroc, positions=positions, widths=0.6, showmeans=True)
colors_vl = ['#4e79a7', '#f28e2b']
for i, (pc, color) in enumerate(zip(parts['bodies'], colors_vl)):
    pc.set_facecolor(color); pc.set_alpha(0.7)
for k in ['cmins', 'cmaxes', 'cbars', 'cmeans']:
    parts[k].set_color('black'); parts[k].set_lw(1.5)
for i, (data, color, pos) in enumerate(zip(data_auroc, colors_vl, positions)):
    ax.scatter(np.full(5, pos) + rng.uniform(-0.05, 0.05, 5), data, color='black', s=30, zorder=3)
ax.set_xticks(positions); ax.set_xticklabels(['k-mer\n+ LR', 'k-mer\n+ SVM'])
ax.set_ylabel('AUROC (5-fold CV)')
ax.set_ylim(0.5, 1.05)
ax.set_title('A. Baseline AUROC comparison\n(n=5 folds; dots = individual folds)')

# Panel B: ROC curves (mean across folds)
ax = axes[1]
skf2 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for clf_pipe, name, color in [(lr_pipe,'k-mer+LR','#4e79a7'),(svm_pipe,'k-mer+SVM','#f28e2b')]:
    from sklearn.metrics import roc_curve
    fpr_list, tpr_list = [], []
    for train_idx, test_idx in skf2.split(X_kmer, labels):
        clf_pipe.fit(X_kmer[train_idx], labels[train_idx])
        proba = clf_pipe.predict_proba(X_kmer[test_idx])[:, 1]
        fpr, tpr, _ = roc_curve(labels[test_idx], proba)
        fpr_list.append(fpr); tpr_list.append(tpr)
    # Interpolate to common FPR grid
    fpr_grid = np.linspace(0, 1, 100)
    tpr_interp = np.array([np.interp(fpr_grid, f, t) for f, t in zip(fpr_list, tpr_list)])
    ax.plot(fpr_grid, tpr_interp.mean(axis=0), color=color, lw=2, label=name)
    ax.fill_between(fpr_grid, tpr_interp.mean(axis=0)-tpr_interp.std(axis=0),
                      tpr_interp.mean(axis=0)+tpr_interp.std(axis=0), color=color, alpha=0.2)
ax.plot([0,1],[0,1],'k--',lw=0.8,label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('B. ROC curves (mean ± SD\nacross 5 folds)')
ax.legend(fontsize=9)

# Panel C: k-mer frequency heatmap (positive vs negative)
ax = axes[2]
mean_pos = X_kmer[labels==1].mean(axis=0)
mean_neg = X_kmer[labels==0].mean(axis=0)
top_diff_idx = np.argsort(np.abs(mean_pos - mean_neg))[-20:][::-1]
diff_matrix = np.vstack([mean_pos[top_diff_idx], mean_neg[top_diff_idx]])
im = ax.imshow(diff_matrix, aspect='auto', cmap='RdBu_r')
plt.colorbar(im, ax=ax, shrink=0.8, label='Mean frequency')
ax.set_yticks([0, 1]); ax.set_yticklabels(['Positive', 'Negative'])
ax.set_xlabel('Top 20 most discriminative 4-mers')
ax.set_xticks([])
ax.set_title('C. k-mer frequency: positive vs negative\n(top 20 by absolute difference)')

plt.suptitle('Module 20 CP02: Problem Setup and Baselines', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('cp02_baselines.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 12 — Reflection

> *[Why does the k-mer+SVM baseline use 5-fold cross-validation rather than
> a single train/test split? What would happen to the reported AUROC if you
> optimized the SVM hyperparameters (C) on the same fold used for evaluation?]*

---
*Next: `06_cp02_cnn_implementation.ipynb`*